# RAG Chatbot - Retrieval Augmented Generation Testing

This notebook scans a Dataiku managed folder for documents, chunks them, builds a vector index, and provides an interactive chatbot interface for testing RAG.

## 1. Configuration

All configurable parameters are set here. LLM selection is driven by Dataiku project variables.

In [ ]:
import dataiku
import os
import json
import re
import io
import numpy as np
from pathlib import PurePosixPath

# ---------------------------------------------------------------------------
# Configuration - edit these or set via Dataiku project variables
# ---------------------------------------------------------------------------
project_variables = dataiku.get_custom_variables()

# Managed folder ID that contains the documents to index
FOLDER_ID = project_variables.get("RAG_FOLDER_ID", "documents")

# LLM configuration (Dataiku LLM Mesh)
LLM_ID = project_variables.get("RAG_LLM_ID", "openai:gpt-4o-mini")
EMBEDDING_LLM_ID = project_variables.get("RAG_EMBEDDING_LLM_ID", "openai:text-embedding-3-small")

# Chunking parameters
CHUNK_SIZE = int(project_variables.get("RAG_CHUNK_SIZE", 1000))
CHUNK_OVERLAP = int(project_variables.get("RAG_CHUNK_OVERLAP", 200))

# Retrieval parameters
TOP_K = int(project_variables.get("RAG_TOP_K", 5))

# Supported file extensions
SUPPORTED_EXTENSIONS = {".txt", ".md", ".csv", ".pdf", ".docx", ".json", ".html"}

print(f"Folder ID:        {FOLDER_ID}")
print(f"LLM ID:           {LLM_ID}")
print(f"Embedding LLM ID: {EMBEDDING_LLM_ID}")
print(f"Chunk size:       {CHUNK_SIZE}")
print(f"Chunk overlap:    {CHUNK_OVERLAP}")
print(f"Top-K retrieval:  {TOP_K}")

## 2. Document Scanning

Recursively scan the Dataiku managed folder and extract text from supported file types.

In [ ]:
# ---------------------------------------------------------------------------
# Text extraction helpers
# ---------------------------------------------------------------------------

def extract_text_txt(raw_bytes):
    """Extract text from plain text / markdown / csv files."""
    return raw_bytes.decode("utf-8", errors="replace")


def extract_text_pdf(raw_bytes):
    """Extract text from a PDF using PyPDF2."""
    try:
        import PyPDF2
    except ImportError:
        raise ImportError("PyPDF2 is required for PDF support. Install it with: pip install PyPDF2")
    reader = PyPDF2.PdfReader(io.BytesIO(raw_bytes))
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages)


def extract_text_docx(raw_bytes):
    """Extract text from a DOCX file using python-docx."""
    try:
        import docx
    except ImportError:
        raise ImportError("python-docx is required for DOCX support. Install it with: pip install python-docx")
    doc = docx.Document(io.BytesIO(raw_bytes))
    return "\n\n".join(p.text for p in doc.paragraphs if p.text.strip())


def extract_text_json(raw_bytes):
    """Pretty-print JSON content as text."""
    data = json.loads(raw_bytes.decode("utf-8", errors="replace"))
    return json.dumps(data, indent=2, ensure_ascii=False)


def extract_text_html(raw_bytes):
    """Strip HTML tags and return plain text."""
    text = raw_bytes.decode("utf-8", errors="replace")
    text = re.sub(r"<script[^>]*>[\s\S]*?</script>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<style[^>]*>[\s\S]*?</style>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


EXTRACTORS = {
    ".txt": extract_text_txt,
    ".md": extract_text_txt,
    ".csv": extract_text_txt,
    ".pdf": extract_text_pdf,
    ".docx": extract_text_docx,
    ".json": extract_text_json,
    ".html": extract_text_html,
}

In [ ]:
# ---------------------------------------------------------------------------
# Scan the managed folder
# ---------------------------------------------------------------------------
folder = dataiku.Folder(FOLDER_ID)
all_paths = folder.list_paths_in_partition()

documents = []  # list of {"path": str, "ext": str, "text": str, "size": int}
skipped = []

for path in sorted(all_paths):
    ext = PurePosixPath(path).suffix.lower()
    if ext not in SUPPORTED_EXTENSIONS:
        skipped.append((path, "unsupported type"))
        continue
    try:
        with folder.get_download_stream(path) as stream:
            raw_bytes = stream.read()
        text = EXTRACTORS[ext](raw_bytes)
        if not text.strip():
            skipped.append((path, "empty content"))
            continue
        documents.append({
            "path": path,
            "ext": ext,
            "text": text,
            "size": len(raw_bytes),
        })
    except Exception as e:
        skipped.append((path, str(e)))

print(f"Loaded {len(documents)} documents, skipped {len(skipped)} files.")
print()

# Summary table
print(f"{'Path':<60} {'Type':<8} {'Size':>10} {'Text Len':>10}")
print("-" * 90)
for doc in documents:
    print(f"{doc['path']:<60} {doc['ext']:<8} {doc['size']:>10,} {len(doc['text']):>10,}")

if skipped:
    print(f"\nSkipped files:")
    for path, reason in skipped:
        print(f"  {path}: {reason}")

## 3. Document Chunking

Split documents into overlapping chunks for retrieval.

In [ ]:
# ---------------------------------------------------------------------------
# Chunking logic
# ---------------------------------------------------------------------------

def chunk_text(text, chunk_size, chunk_overlap):
    """Split text into overlapping chunks, preferring to break on paragraph/sentence boundaries."""
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = start + chunk_size

        if end < text_len:
            # Try to break at a paragraph boundary
            break_point = text.rfind("\n\n", start, end)
            if break_point == -1 or break_point <= start:
                # Fall back to sentence boundary
                break_point = text.rfind(". ", start, end)
            if break_point == -1 or break_point <= start:
                # Fall back to word boundary
                break_point = text.rfind(" ", start, end)
            if break_point > start:
                end = break_point + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append({"text": chunk, "start": start, "end": end})

        start = end - chunk_overlap
        if start >= text_len:
            break
        # Avoid infinite loop if overlap >= chunk produced
        if len(chunks) > 0 and start <= chunks[-1]["start"]:
            start = end

    return chunks


# Build chunk corpus
all_chunks = []  # list of {"text": str, "source": str, "chunk_idx": int, "start": int, "end": int}

for doc in documents:
    doc_chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for idx, c in enumerate(doc_chunks):
        all_chunks.append({
            "text": c["text"],
            "source": doc["path"],
            "chunk_idx": idx,
            "start": c["start"],
            "end": c["end"],
        })

# Statistics
chunk_lengths = [len(c["text"]) for c in all_chunks]
print(f"Total chunks: {len(all_chunks)}")
if chunk_lengths:
    print(f"Avg chunk length: {np.mean(chunk_lengths):.0f} chars")
    print(f"Min: {np.min(chunk_lengths)}, Max: {np.max(chunk_lengths)}, Median: {np.median(chunk_lengths):.0f}")
    print()
    # Per-document breakdown
    from collections import Counter
    source_counts = Counter(c["source"] for c in all_chunks)
    print(f"{'Source':<60} {'Chunks':>8}")
    print("-" * 70)
    for src, count in source_counts.most_common():
        print(f"{src:<60} {count:>8}")

## 4. Embedding & Vector Index

Generate embeddings using the Dataiku LLM Mesh and build an in-memory vector index.

In [ ]:
# ---------------------------------------------------------------------------
# Embedding via Dataiku LLM Mesh
# ---------------------------------------------------------------------------

client = dataiku.api_client()
project = client.get_default_project()


def get_embeddings(texts, batch_size=20):
    """Get embeddings for a list of texts using the Dataiku LLM Mesh embedding model."""
    embedding_model = project.get_llm(EMBEDDING_LLM_ID)
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        emb_query = embedding_model.new_embeddings()
        for t in batch:
            emb_query.add_text(t)
        resp = emb_query.execute()
        all_embeddings.extend([e["embedding"] for e in resp.get_embeddings()])
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks", end="\r")

    print()
    return np.array(all_embeddings, dtype=np.float32)


print("Generating embeddings for all chunks...")
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = get_embeddings(chunk_texts)
print(f"Embedding matrix shape: {chunk_embeddings.shape}")

# Normalize for cosine similarity
norms = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1
chunk_embeddings_normed = chunk_embeddings / norms
print("Vector index ready.")

## 5. Retrieval

Given a query, find the top-K most relevant chunks.

In [ ]:
# ---------------------------------------------------------------------------
# Retrieval function
# ---------------------------------------------------------------------------

def retrieve(query, top_k=TOP_K):
    """Embed the query and return the top-k most similar chunks."""
    q_emb = get_embeddings([query])
    q_normed = q_emb[0] / (np.linalg.norm(q_emb[0]) or 1)

    # Cosine similarity via dot product (vectors are pre-normalized)
    similarities = chunk_embeddings_normed @ q_normed
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "chunk": all_chunks[idx],
            "score": float(similarities[idx]),
        })
    return results


# Quick test
if all_chunks:
    test_results = retrieve("test query")
    print(f"Retrieval test: got {len(test_results)} results")
    for r in test_results[:3]:
        print(f"  score={r['score']:.4f}  source={r['chunk']['source']}  chunk={r['chunk']['chunk_idx']}")

## 6. RAG Generation

Compose a prompt with retrieved context and generate an answer using the configured LLM.

In [ ]:
# ---------------------------------------------------------------------------
# RAG answer generation
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a helpful assistant that answers questions based on the provided context documents.
Always ground your answers in the context provided. If the context does not contain enough
information to answer the question, say so clearly. Cite the source file for key facts."""


def build_context_block(retrieved):
    """Format retrieved chunks into a context block for the LLM."""
    parts = []
    for i, r in enumerate(retrieved, 1):
        source = r["chunk"]["source"]
        score = r["score"]
        text = r["chunk"]["text"]
        parts.append(f"[Source {i}: {source} (relevance: {score:.3f})]\n{text}")
    return "\n\n---\n\n".join(parts)


def rag_answer(question, conversation_history=None, top_k=TOP_K):
    """Retrieve context and generate an answer."""
    retrieved = retrieve(question, top_k=top_k)
    context_block = build_context_block(retrieved)

    llm = project.get_llm(LLM_ID)
    completion = llm.new_completion()
    completion.with_message(SYSTEM_PROMPT, role="system")

    # Include conversation history for multi-turn
    if conversation_history:
        for msg in conversation_history:
            completion.with_message(msg["content"], role=msg["role"])

    user_message = f"""Context documents:
---
{context_block}
---

Question: {question}"""

    completion.with_message(user_message, role="user")
    resp = completion.execute()
    answer = resp.text

    return {
        "answer": answer,
        "sources": retrieved,
    }


print("RAG pipeline ready.")

## 7. Interactive Chatbot Interface

A simple in-notebook chatbot UI using `ipywidgets`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------------------------------------------------------------------------
# Chat state
# ---------------------------------------------------------------------------
conversation_history = []
chat_log_entries = []  # for display: list of {"role": str, "content": str, "sources": list|None}

# ---------------------------------------------------------------------------
# UI components
# ---------------------------------------------------------------------------
chat_output = widgets.Output(layout=widgets.Layout(
    width="100%",
    min_height="300px",
    max_height="600px",
    overflow_y="auto",
    border="1px solid #ccc",
    padding="10px",
))

user_input = widgets.Text(
    placeholder="Ask a question about your documents...",
    layout=widgets.Layout(width="80%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width="10%"),
)

clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    layout=widgets.Layout(width="10%"),
)

status_label = widgets.Label(value="Ready.")


def render_chat():
    """Re-render the chat output area."""
    with chat_output:
        clear_output(wait=True)
        for entry in chat_log_entries:
            if entry["role"] == "user":
                display(HTML(
                    f'<div style="margin:8px 0; padding:8px 12px; '
                    f'background:#e3f2fd; border-radius:8px;">'
                    f'<b>You:</b> {entry["content"]}</div>'
                ))
            else:
                source_html = ""
                if entry.get("sources"):
                    source_items = "".join(
                        f"<li>{r['chunk']['source']} (chunk {r['chunk']['chunk_idx']}, "
                        f"score: {r['score']:.3f})</li>"
                        for r in entry["sources"]
                    )
                    source_html = (
                        f'<details style="margin-top:6px; font-size:0.85em; color:#666;">'
                        f'<summary>Sources ({len(entry["sources"])})</summary>'
                        f'<ul>{source_items}</ul></details>'
                    )
                answer_html = entry["content"].replace("\n", "<br>")
                display(HTML(
                    f'<div style="margin:8px 0; padding:8px 12px; '
                    f'background:#f1f8e9; border-radius:8px;">'
                    f'<b>Assistant:</b><br>{answer_html}{source_html}</div>'
                ))


def on_send(button=None):
    """Handle send button click or Enter key."""
    question = user_input.value.strip()
    if not question:
        return

    user_input.value = ""
    chat_log_entries.append({"role": "user", "content": question, "sources": None})
    render_chat()

    status_label.value = "Thinking..."
    try:
        result = rag_answer(question, conversation_history=conversation_history)
        answer = result["answer"]
        sources = result["sources"]

        # Update conversation history for multi-turn
        conversation_history.append({"role": "user", "content": question})
        conversation_history.append({"role": "assistant", "content": answer})

        chat_log_entries.append({"role": "assistant", "content": answer, "sources": sources})
        status_label.value = "Ready."
    except Exception as e:
        chat_log_entries.append({"role": "assistant", "content": f"Error: {e}", "sources": None})
        status_label.value = f"Error: {e}"

    render_chat()


def on_clear(button=None):
    """Clear the conversation."""
    conversation_history.clear()
    chat_log_entries.clear()
    render_chat()
    status_label.value = "Conversation cleared."


# Wire up events
send_button.on_click(on_send)
user_input.on_submit(on_send)
clear_button.on_click(on_clear)

# Layout
input_row = widgets.HBox([user_input, send_button, clear_button])
chat_ui = widgets.VBox([chat_output, input_row, status_label])

display(HTML(f"<p><b>Config:</b> LLM={LLM_ID} | Embeddings={EMBEDDING_LLM_ID} | "
             f"Chunks={len(all_chunks)} | Top-K={TOP_K}</p>"))
display(chat_ui)